In [147]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import plotly.express as px
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
pd.set_option('display.max_columns', None)
sns.set(font_scale=1.2)

print("1. Загрузка данных")
df = pd.read_csv('MoviesOnStreamingPlatforms.csv')

1. Загрузка данных


Быстрый обзор данных (Pandas)
• df.head(), df.tail(), df.shape.
• df.info() и df.describe() (отдельно покажите .describe(include="object") для строк).
• Проверки качества: isnull().sum(), поиск дубликатов, проверка адекватности типов данных.

In [148]:
print("\nПервые 10 объектов:")
df.head(10)


Первые 10 объектов:


,Unnamed: 0,ID,Title,Year,Age,Rotten Tomatoes,Netflix,Hulu,Prime Video,Disney+,Type
0,0,1,The Irishman,2019,18+,98/100,1,0,0,0,0
1,1,2,Dangal,2016,7+,97/100,1,0,0,0,0
2,2,3,David Attenborough: A Life on Our Planet,2020,7+,95/100,1,0,0,0,0
3,3,4,Lagaan: Once Upon a Time in India,2001,7+,94/100,1,0,0,0,0
4,4,5,Roma,2018,18+,94/100,1,0,0,0,0
5,5,6,To All the Boys I've Loved Before,2018,13+,94/100,1,0,0,0,0
6,6,7,The Social Dilemma,2020,13+,93/100,1,0,0,0,0
7,7,8,Okja,2017,13+,92/100,1,0,0,0,0
8,8,9,The Ballad of Buster Scruggs,2018,16+,92/100,1,0,0,0,0
9,9,10,The Trial of the Chicago 7,2020,18+,92/100,1,0,0,0,0


In [149]:
print("\nПоследние 10 объектов:")
df.tail(10)


Последние 10 объектов:


,Unnamed: 0,ID,Title,Year,Age,Rotten Tomatoes,Netflix,Hulu,Prime Video,Disney+,Type
9505,9505,9506,Great Shark Chow Down,2019,7+,14/100,0,0,0,1,0
9506,9506,9507,In Beaver Valley,1950,NaN,14/100,0,0,0,1,0
9507,9507,9508,Texas Storm Squad,2020,13+,14/100,0,0,0,1,0
9508,9508,9509,What the Shark?,2020,13+,14/100,0,0,0,1,0
9509,9509,9510,Built for Mars: The Perseverance Rover,2021,NaN,14/100,0,0,0,1,0
9510,9510,9511,Most Wanted Sharks,2020,NaN,14/100,0,0,0,1,0
9511,9511,9512,Doc McStuffins: The Doc Is In,2020,NaN,13/100,0,0,0,1,0
9512,9512,9513,Ultimate Viking Sword,2019,NaN,13/100,0,0,0,1,0
9513,9513,9514,Hunt for the Abominable Snowman,2011,NaN,10/100,0,0,0,1,0
9514,9514,9515,Women of Impact: Changing the World,2019,7+,10/100,0,0,0,1,0


In [150]:
print("\nРазмер датасета: "+str(df.shape))


Размер датасета: (9515, 11)


In [151]:
print("\nОбщая информация о датасете:")
df.info()


Общая информация о датасете:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9515 entries, 0 to 9514
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Unnamed: 0       9515 non-null   int64 
 1   ID               9515 non-null   int64 
 2   Title            9515 non-null   object
 3   Year             9515 non-null   int64 
 4   Age              5338 non-null   object
 5   Rotten Tomatoes  9508 non-null   object
 6   Netflix          9515 non-null   int64 
 7   Hulu             9515 non-null   int64 
 8   Prime Video      9515 non-null   int64 
 9   Disney+          9515 non-null   int64 
 10  Type             9515 non-null   int64 
dtypes: int64(8), object(3)
memory usage: 817.8+ KB


In [152]:
print("\nСтатистика по числовым признакам:")
df.describe().T


Статистика по числовым признакам:


,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,9515.0,4757.000000,2746.888239,0.0,2378.5,4757.0,7135.5,9514.0
ID,9515.0,4758.000000,2746.888239,1.0,2379.5,4758.0,7136.5,9515.0
Year,9515.0,2007.422386,19.130367,1914.0,2006.0,2015.0,2018.0,2021.0
Netflix,9515.0,0.388334,0.487397,0.0,0.0,0.0,1.0,1.0
Hulu,9515.0,0.110037,0.312952,0.0,0.0,0.0,0.0,1.0
Prime Video,9515.0,0.432265,0.495417,0.0,0.0,0.0,1.0,1.0
Disney+,9515.0,0.096900,0.295837,0.0,0.0,0.0,0.0,1.0
Type,9515.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


In [153]:
print("\nСтатистика по различным типам данных:")
print("\nСтатистика по колонкам типа 'object':")
df.describe(include="object").T


Статистика по различным типам данных:

Статистика по колонкам типа 'object':


,count,unique,top,freq
Title,9515,9515,The Irishman,1
Age,5338,5,18+,2276
Rotten Tomatoes,9508,85,44/100,311


In [154]:
print("\nСтатистика по колонкам типа 'int64':")
df.describe(include="int64").T


Статистика по колонкам типа 'int64':


,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,9515.0,4757.000000,2746.888239,0.0,2378.5,4757.0,7135.5,9514.0
ID,9515.0,4758.000000,2746.888239,1.0,2379.5,4758.0,7136.5,9515.0
Year,9515.0,2007.422386,19.130367,1914.0,2006.0,2015.0,2018.0,2021.0
Netflix,9515.0,0.388334,0.487397,0.0,0.0,0.0,1.0,1.0
Hulu,9515.0,0.110037,0.312952,0.0,0.0,0.0,0.0,1.0
Prime Video,9515.0,0.432265,0.495417,0.0,0.0,0.0,1.0,1.0
Disney+,9515.0,0.096900,0.295837,0.0,0.0,0.0,0.0,1.0
Type,9515.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


In [155]:
print("\nОбработка пропущенных значений:")

print("\nКоличество пропущенных значений по столбцам:")
missing_values = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'missing_values': missing_values,
                          'percent_missing': missing_percent.round(2)})
print(missing_df[missing_df['missing_values'] > 0])


Обработка пропущенных значений:

Количество пропущенных значений по столбцам:
                 missing_values  percent_missing
Age                        4177            43.90
Rotten Tomatoes               7             0.07


В качестве заполнения используем константу, так как нельзя вычислить среднее значение или медиану пропущенных значений и данное заполнение имеет правильное логическое значение. Также мы не используем моду, так как тогда в измененном датасете могут быть ошибки

In [156]:
if missing_df['missing_values'].sum() > 0:
    if 'Age' in df.columns and df['Age'].isnull().sum()>0:
        df['Age'] = df['Age'].fillna("0+")
        print("Заполнили пропущенные значения в 'Age' значением '0+'")
    if 'Rotten Tomatoes' in df.columns and df['Rotten Tomatoes'].isnull().sum()>0:
        df['Rotten Tomatoes'] = df['Rotten Tomatoes'].fillna("No ratings")
        print("Заполнили пропущенные значения в 'Rotten Tomatoes' значением 'No raitings'")

Заполнили пропущенные значения в 'Age' значением '0+'
Заполнили пропущенные значения в 'Rotten Tomatoes' значением 'No raitings'


In [157]:
print("Проверка на дубликаты:")
duplicates = df.duplicated()
print("Кол-во дубликатов: "+str(duplicates.sum()))
duplicates_clean=df.drop_duplicates()

Проверка на дубликаты:
Кол-во дубликатов: 0


In [158]:
print("\nПроверка на адекватность значений:")
print("\n1. Уникальность ID")
if not df['ID'].is_unique:
    print("Кол-во дубликатов ID:"+str(df['ID'].duplicates.sum()))
else:
    print("Дубликатов ID нет")
print("\n2. Адекватность года выпуска")
unvalid_years = (df['Year']>=2027) | (df['Year']<=1880)
print("Кол-во записей с неккоректным годом: "+str(unvalid_years.sum()))
print("\n3. Адекватность возрастного рейтинга")
df['Age']= df['Age'].replace('all', '0+')
print("Уникальные значения 'Age':", df['Age'].unique())
unexpected_age = ~df['Age'].isin(['18+', '16+', '13+', '7+', '0+'])
if unexpected_age.any():
    print("Найдены нестандартные значения Age: ", df.loc[unexpected_age, 'Age'].unique())
else:
    print("Нестандартных значений не найдено")
print("\n4. Адекватность рейтинга Rotten Tomatoes")
df['RT-Score'] = df['Rotten Tomatoes'].str.extract(r'(\d+)/100').astype('Int64')
invalid_score = ~(( (df['RT-Score'] >= 0) & (df['RT-Score'] <= 100) ) | (df['RT-Score'] == 'No raitings'))
print("Некорректные значения Rotten Tomatoes: "+str(invalid_score.sum()))
print("\n5. Проверка бинарных перменных:")
binary = ['Netflix', 'Hulu', 'Prime Video', 'Disney+', 'Type']
for col in binary:
    invalid = ~df[col].isin([0, 1])
    if invalid.any():
        print("В колонке "+col+" найдены значения, отличныя от 0 и 1: "+df.loc[invalid, col].unique())
    else:
        print("В колонке "+col+" все значения корректны")



Проверка на адекватность значений:

1. Уникальность ID
Дубликатов ID нет

2. Адекватность года выпуска
Кол-во записей с неккоректным годом: 0

3. Адекватность возрастного рейтинга
Уникальные значения 'Age': ['18+' '7+' '13+' '16+' '0+']
Нестандартных значений не найдено

4. Адекватность рейтинга Rotten Tomatoes
Некорректные значения Rotten Tomatoes: 0

5. Проверка бинарных перменных:
В колонке Netflix все значения корректны
В колонке Hulu все значения корректны
В колонке Prime Video все значения корректны
В колонке Disney+ все значения корректны
В колонке Type все значения корректны


B) Пропуски и очистка
• Покажите стратегии dropna() и fillna(). Заполняйте средним, медианой, константой — и обязательно обоснуйте выбор текстовым комментарием.
• Отдельно покажите, как считаете моду (mode) и заполняете ею пропуски в категориальных колонках.

Почему использовали именно такую стратегию замены пропущенных значений было описано выше. Ниже вычислим моду переменных:

In [159]:
print("\nМода для каждой перменной (где имеет смысл):")
print("\nYear: "+str(df['Year'].mode()[0]))
print("Age: "+str(df['Age'].mode()[0]))
print("Rotten Tomatoes: "+str(df['Rotten Tomatoes'].mode()[0]))
print("Netflix: "+str(df['Netflix'].mode()[0]))
print("Hulu: "+str(df['Hulu'].mode()[0]))
print("Prime Video: "+str(df['Prime Video'].mode()[0]))
print("Disney+ : "+str(df['Disney+'].mode()[0]))


Мода для каждой перменной (где имеет смысл):

Year: 2019
Age: 0+
Rotten Tomatoes: 44/100
Netflix: 0
Hulu: 0
Prime Video: 0
Disney+ : 0


 C) Расширенная статистика
• Для числовых колонок выведите: min, max, mean, median, mode.
• Посчитайте percentile/quantile (например, 5, 25, 50, 75, 95 перцентили).
🧠 Самостоятельно изучите и посчитайте: дисперсию (variance), асимметрию (skewness) и эксцесс (kurtosis). Попробуйте объяснить, что они значат для ваших данных.

In [160]:
print("\nmin, max, mean, median, mode для числовых колонок (и Rotten Tomatoes):")
numeric_cols = df.select_dtypes("int64").columns.to_list()
numeric_cols.remove('ID')
numeric_cols.remove('Unnamed: 0')
print("\nМинимальные значения:")
for col in numeric_cols:
    print("Минимальное значение в колонке "+col+": "+str(df[col].min()))
print("\nМаксимальные значения:")
for col in numeric_cols:
    print("Максимальное значение в колонке "+col+": "+str(df[col].max()))


min, max, mean, median, mode для числовых колонок (и Rotten Tomatoes):

Минимальные значения:
Минимальное значение в колонке Year: 1914
Минимальное значение в колонке Netflix: 0
Минимальное значение в колонке Hulu: 0
Минимальное значение в колонке Prime Video: 0
Минимальное значение в колонке Disney+: 0
Минимальное значение в колонке Type: 0
Минимальное значение в колонке RT-Score: 10

Максимальные значения:
Максимальное значение в колонке Year: 2021
Максимальное значение в колонке Netflix: 1
Максимальное значение в колонке Hulu: 1
Максимальное значение в колонке Prime Video: 1
Максимальное значение в колонке Disney+: 1
Максимальное значение в колонке Type: 0
Максимальное значение в колонке RT-Score: 98


In [161]:
print("\nСредние арифметические значения:")
for col in numeric_cols:
    print("Среднее арифметическое значение в колонке "+col+": "+str(df[col].mean()))
print("\nМедиана:")
for col in numeric_cols:
    print("Медиана в колонке "+col+": "+str(df[col].median()))
print("\nМода:")
for col in numeric_cols:
    print("Мода в колонке "+col+": "+str(df[col].mode()[0]))


Средние арифметические значения:
Среднее арифметическое значение в колонке Year: 2007.4223857067789
Среднее арифметическое значение в колонке Netflix: 0.3883342091434577
Среднее арифметическое значение в колонке Hulu: 0.11003678402522334
Среднее арифметическое значение в колонке Prime Video: 0.432264844981608
Среднее арифметическое значение в колонке Disney+: 0.09689963215974777
Среднее арифметическое значение в колонке Type: 0.0
Среднее арифметическое значение в колонке RT-Score: 53.545014724442574

Медиана:
Медиана в колонке Year: 2015.0
Медиана в колонке Netflix: 0.0
Медиана в колонке Hulu: 0.0
Медиана в колонке Prime Video: 0.0
Медиана в колонке Disney+: 0.0
Медиана в колонке Type: 0.0
Медиана в колонке RT-Score: 52.0

Мода:
Мода в колонке Year: 2019
Мода в колонке Netflix: 0
Мода в колонке Hulu: 0
Мода в колонке Prime Video: 0
Мода в колонке Disney+: 0
Мода в колонке Type: 0
Мода в колонке RT-Score: 44


In [162]:
print("Вычисление percentile/quantile")
percentile_df = df[numeric_cols].quantile([0.05, 0.25, 0.5, 0.75, 0.95])
print(percentile_df.T)

Вычисление percentile/quantile
               0.05    0.25    0.50    0.75    0.95
Year         1956.0  2006.0  2015.0  2018.0  2020.0
Netflix         0.0     0.0     0.0     1.0     1.0
Hulu            0.0     0.0     0.0     0.0     1.0
Prime Video     0.0     0.0     0.0     1.0     1.0
Disney+         0.0     0.0     0.0     0.0     1.0
Type            0.0     0.0     0.0     0.0     0.0
RT-Score         36      44      52      62      77


In [163]:
print("\nДисперсия:")
variance_all = df[numeric_cols].var()
print(variance_all.T)


Дисперсия:
Year            365.97093
Netflix          0.237556
Hulu             0.097939
Prime Video      0.245438
Disney+          0.087519
Type                  0.0
RT-Score       174.178577
dtype: Float64


In [164]:
print("\nАсимметрия:")
skew_all = df[numeric_cols].skew()
print(skew_all.T)


Асимметрия:
Year          -2.359848
Netflix        0.458309
Hulu           2.492684
Prime Video    0.273505
Disney+        2.725728
Type                0.0
RT-Score       0.192554
dtype: Float64


In [165]:
print("\nЭксцесс:")
kurt_all = df[numeric_cols].kurtosis()
print(kurt_all.T)


Эксцесс:
Year           5.147874
Netflix       -1.790329
Hulu           4.214357
Prime Video     -1.9256
Disney+        5.430737
Type                0.0
RT-Score        0.34379
dtype: Float64


Информативными данными являются только Year и RT-Score.

Дисперсия:
Для года выпуска дисперсия показывает насколько разнообразны годы выпуска фильмов. Год выпуска достаточно отклоняется от среднего, так как годы выпуска фильмов корелируются от 1914 до 2021 года. Похожее можно сказать и про RT-Score, но отклонение не такое большое (значения от 10 до 98)

Ассиметрия:
Для года выпуска ассиметрия показывает насколько много фильмов выпускали за последние годы. Ассиметрия для фильмов отрицательная - значит за последние годы снимали больше фильмов. Но значение по модулю не такое большое и кол-во фильмов по годам все же не значительно больше. Результат ассиметрии для рейтинга показывает, что фильмов с высокими и низкими оценкам примерно одинаковое количество.

Эксцесс:
Эксцесс для года выпуска положительный - значит в какой-то год фильмов снимали значительно больше. Эксцесс по рейтингу примерно равен нулю - значит распределение оценок близко к нормальному.

D) Фичи: Энкодинг и Инжиниринг (Feature Engineering)
• Сделайте кодирование категорий: OneHotEncoder (или pd.get_dummies) — обязательно.
• Если уместно: Label Encoding или Target Encoding. Покажите датафрейм до/после.
 🧠 Самостоятельно изучите: попробуйте применить Feature Hashing (Hashing Encoder) для признаков с большим числом уникальных значений.
 🧠 Генерация новых фич: создайте 2-3 новых признака. Например, перемножьте/поделите две существующие колонки, вытащите месяц из даты или сгруппируйте редкие категории в одну "Other".

In [166]:
cols_onehot = ['Age']
df_onehot = pd.get_dummies(df, columns=cols_onehot, prefix=cols_onehot, drop_first=False)
print("\nДатасет после OneHotEncoder:")
print(df_onehot.head())


Датасет после OneHotEncoder:
   Unnamed: 0  ID                                     Title  Year  \
0           0   1                              The Irishman  2019   
1           1   2                                    Dangal  2016   
2           2   3  David Attenborough: A Life on Our Planet  2020   
3           3   4         Lagaan: Once Upon a Time in India  2001   
4           4   5                                      Roma  2018   

  Rotten Tomatoes  Netflix  Hulu  Prime Video  Disney+  Type  RT-Score  \
0          98/100        1     0            0        0     0        98   
1          97/100        1     0            0        0     0        97   
2          95/100        1     0            0        0     0        95   
3          94/100        1     0            0        0     0        94   
4          94/100        1     0            0        0     0        94   

   Age_0+  Age_13+  Age_16+  Age_18+  Age_7+  
0   False    False    False     True   False  
1   False    Fal

In [167]:
from sklearn.preprocessing import LabelEncoder
df_label = df.copy()
LE = LabelEncoder()
df_label['Age-Label'] = LE.fit_transform(df_label['Age'])
print("\nLabel Encoding для Age:")
print(df_label.head())


Label Encoding для Age:
   Unnamed: 0  ID                                     Title  Year  Age  \
0           0   1                              The Irishman  2019  18+   
1           1   2                                    Dangal  2016   7+   
2           2   3  David Attenborough: A Life on Our Planet  2020   7+   
3           3   4         Lagaan: Once Upon a Time in India  2001   7+   
4           4   5                                      Roma  2018  18+   

  Rotten Tomatoes  Netflix  Hulu  Prime Video  Disney+  Type  RT-Score  \
0          98/100        1     0            0        0     0        98   
1          97/100        1     0            0        0     0        97   
2          95/100        1     0            0        0     0        95   
3          94/100        1     0            0        0     0        94   
4          94/100        1     0            0        0     0        94   

   Age-Label  
0          3  
1          4  
2          4  
3          4  
4         

In [168]:
from sklearn.preprocessing import TargetEncoder

df_label = df.copy()
TE = TargetEncoder()
median_rating = df_label['RT-Score'].median()
df_label['RT_score_filled'] = df_label['RT-Score'].fillna(median_rating)
df_label['high_raiting'] = (df_label['RT_score_filled'] > 80).astype(int)
df_label['AgeTarget'] = TE.fit_transform(df_label[['Age']], df_label['high_raiting'])
print("\nTarget Encoding для Age (среднее high_raiting):")
print(df_label[['Age', 'high_raiting', 'AgeTarget']].drop_duplicates().head(10))


Target Encoding для Age (среднее high_raiting):
    Age  high_raiting  AgeTarget
0   18+             1   0.047449
1    7+             1   0.045963
2    7+             1   0.049739
4   18+             1   0.050636
5   13+             1   0.063050
6   13+             1   0.070176
7   13+             1   0.070852
8   16+             1   0.041408
9   18+             1   0.053022
10  18+             1   0.048523
